# SARSA

SARSA は「次に実際に選ぶ行動」で更新する on-policy 手法です。探索を含んだ方策そのものを改善するので、リスクの高い行動を避けたい場面で挙動が安定しやすい特徴があります。


## 参考動画
- [対応動画 1](https://www.youtube.com/watch?v=K69r80Ih544)
@[youtube](https://www.youtube.com/watch?v=K69r80Ih544)

## 参考リンク
- https://www.youtube.com/watch?v=K69r80Ih544


## 背景と目的

Q学習は次状態で最大価値を参照するため、探索中でも強気な更新になりやすいです。

SARSA は実際に選んだ次行動を参照するため、探索ノイズを含んだ現実的な期待値に近づきます。

更新式の違いがどのような方策差につながるかを、同じ環境設定で比較します。


In [ ]:
import random
from collections import defaultdict

random.seed(7)

gamma = 0.95
alpha = 0.2
epsilon = 0.2
actions = ['safe', 'risky']


def step(state, action):
    # 2段階のエピソードにして、s' での行動選択が効くようにする
    if state == 's0':
        if action == 'safe':
            return 's1', 0.2
        return 's1', (0.7 if random.random() < 0.5 else -0.6)

    if state == 's1':
        if action == 'safe':
            return 'terminal', 0.6
        return 'terminal', (1.2 if random.random() < 0.2 else -1.0)

    return 'terminal', 0.0


def eps_greedy(Q, state):
    if random.random() < epsilon:
        return random.choice(actions)
    qs = [Q[(state, a)] for a in actions]
    return actions[0] if qs[0] >= qs[1] else actions[1]


## 1ステップ更新の式

SARSA:

\[Q(s,a) \leftarrow Q(s,a) + \alpha \{r + \gamma Q(s',a') - Q(s,a)\}\]

Q学習:

\[Q(s,a) \leftarrow Q(s,a) + \alpha \{r + \gamma \max_{a'}Q(s',a') - Q(s,a)\}\]

差は目標値の作り方だけですが、学習結果の方策が変わります。


In [ ]:
Q_sarsa = defaultdict(float)
Q_q = defaultdict(float)

for _ in range(800):
    # SARSA episode
    s = 's0'
    a = eps_greedy(Q_sarsa, s)
    while s != 'terminal':
        s_next, r = step(s, a)
        if s_next == 'terminal':
            target = r
            Q_sarsa[(s, a)] += alpha * (target - Q_sarsa[(s, a)])
            break
        a_next = eps_greedy(Q_sarsa, s_next)
        target = r + gamma * Q_sarsa[(s_next, a_next)]
        Q_sarsa[(s, a)] += alpha * (target - Q_sarsa[(s, a)])
        s, a = s_next, a_next

    # Q-learning episode
    s = 's0'
    while s != 'terminal':
        a = eps_greedy(Q_q, s)
        s_next, r = step(s, a)
        if s_next == 'terminal':
            target = r
        else:
            target = r + gamma * max(Q_q[(s_next, x)] for x in actions)
        Q_q[(s, a)] += alpha * (target - Q_q[(s, a)])
        s = s_next

print('SARSA Q(s0) =', {a: round(Q_sarsa[('s0', a)], 4) for a in actions})
print('Q-learn Q(s0)=', {a: round(Q_q[('s0', a)], 4) for a in actions})
print('SARSA Q(s1) =', {a: round(Q_sarsa[('s1', a)], 4) for a in actions})
print('Q-learn Q(s1)=', {a: round(Q_q[('s1', a)], 4) for a in actions})


高分散な `risky` を過大評価しにくいのが SARSA の利点です。安全性を重視する制御問題で採用される理由はここにあります。
